# Spark-X2.6-4B-En4B-A1B (`v0.3` / `v03_r2_3phase`) — Standalone 72-Layer Inference & Evaluation Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiTian6641/Spark-X2.6-Recursion/blob/main/SparkX_Loop_v03_En4B_A1B_Inference_Eval_Colab.ipynb)

This standalone **inference & evaluation notebook** loads the trained **`TassP0lar/Spark-X2.6-4B-En4B-A1B`** (`v03_r2_3phase`, Global Step 208, `ce_loss = 0.4142`) checkpoint directly from Hugging Face Hub and runs:
1. **Cell 1**: Environment & Repository Setup (`Spark-X2.6-Recursion`)
2. **Cell 2**: One-Call 72-Layer Model Load (`8.34B` Total / `5.33B` Active) + Trained `mHC` & `MoE-Engram` Gate Telemetry Audit
3. **Cell 3**: **3-Mode Ablation & Collapse Audit** (`72L Full Loop + Engram` vs `72L Loop Only` vs `36L Base Backbone`) on unseen algorithmic synthesis (`dijkstra_grid`)
4. **Cell 4**: **Out-of-Distribution Coding & STEM Verification Suite** ($O(1)$ TTL-LRU Cache + Exact Sliding-Window Substring)
5. **Cell 5**: **Unclamped 32K Agentic 3D WebGL Synthesis** (Out-of-distribution `Neon Horizon 2099` Hovercar Simulator & `Sakura Voxel Garden` with live `<iframe>` rendering)
6. **Cell 6**: **Interactive 72-Layer Prompt Playground** (`ask_sparkx_72l(...)`)

In [ ]:
# Cell 1: Environment & Repository Setup (Inference-Only)
import os, subprocess, sys
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48.0', 'accelerate>=1.2.0', 'safetensors>=0.4.5', 'huggingface_hub>=0.27.0'
], check=True)

import torch
print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | Total VRAM: {props.total_memory / (1024**3):.1f} GB')

In [ ]:
# Cell 2: Load Trained Spark-X2.6-4B-En4B-A1B (72L, 32-Expert Top-7) & Inspect Gate Telemetry
import importlib, os, subprocess, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core, load_inference_v03, sparkx_loop_v03_train_env
importlib.reload(sparkx_loop_v03_core)
importlib.reload(load_inference_v03)
importlib.reload(sparkx_loop_v03_train_env)

from load_inference_v03 import load_sparkx_loop_v03
from sparkx_loop_v03_core import verify_birkhoff_sinkhorn, verify_ngram_causality

HF_REPO_ID = 'TassP0lar/Spark-X2.6-4B-En4B-A1B'
HF_TOKEN = os.environ.get('HF_TOKEN', None)
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get('HF_TOKEN')
except Exception:
    pass

model, tokenizer, ckpt_meta = load_sparkx_loop_v03(
    repo_or_dir=HF_REPO_ID,
    enable_loglinear_kv=True,
    loglinear_exact_threshold=4096,
    loglinear_recent_window=2048,
    loglinear_sink_tokens=32,
    offload_tier1b_cpu=True,
    token=HF_TOKEN,
)

# Inspect learned TwoStreamMHC recurrence gates & 8-Anchor MoE-Engram gates
print('\n' + '=' * 96)
print('LEARNED 72-LAYER mHC RECURRENCE & 8-ANCHOR MoE-ENGRAM TELEMETRY')
print('=' * 96)
p2_attn, p2_ffn, p3_attn, p3_ffn = [], [], [], []
for l_idx in range(model.loop_start, model.loop_end):
    c = model.controllers[str(l_idx)]
    p2_attn.append(float(torch.tanh(c.extra_gain.detach().float()).abs().item()))
    p2_ffn.append(float(torch.tanh(c.ffn_extra_gain.detach().float()).abs().item()))
    if model.p3_start <= l_idx < model.p3_end:
        p3_attn.append(float(torch.tanh(c.p3_extra_gain.detach().float()).abs().item()))
        p3_ffn.append(float(torch.tanh(c.p3_ffn_extra_gain.detach().float()).abs().item()))

print(f'  Pass-2 [4..28)  Mean |tanh(gain)| : Attn={sum(p2_attn)/len(p2_attn):.4f} | FFN={sum(p2_ffn)/len(p2_ffn):.4f}')
print(f'  Pass-3 [12..24) Mean |tanh(gain)| : Attn={sum(p3_attn)/len(p3_attn):.4f} | FFN={sum(p3_ffn)/len(p3_ffn):.4f}')

anchor_names = ['Layer 3 (Prelude)'] + [f'Layer {k} (Core/Coda)' for k in model.backbone_engrams.keys()]
for name, eng in zip(anchor_names, model.all_engram_modules()):
    g_val = float(torch.sigmoid(eng.gain.detach().float()).item())
    temp_val = float(eng.router_log_temp.detach().float().exp().item())
    proj_norm = float(eng.project.weight.detach().float().norm().item())
    sw = torch.softmax(eng.scale_weights.detach().float(), dim=0).tolist()
    print(
        f'  Anchor {name:<19}: gate={g_val:.4f} | proj_norm={proj_norm:6.2f} | '
        f'experts={eng.active_experts}(Top-{eng.active_top_k}) | temp={temp_val:.3f} | '
        f'ngram_scales=[2g:{sw[0]:.2f}, 3g:{sw[1]:.2f}, skip:{sw[2]:.2f}]'
    )

sinkhorn_err = max(verify_birkhoff_sinkhorn(c) for c in model.controllers.values())
causal_err = verify_ngram_causality(model)
print(f'\n  [Mathematical Invariants] Sinkhorn Doubly-Stochastic Err={sinkhorn_err:.2e} | N-Gram Causality Err={causal_err:.2e}')
print('=' * 96)

In [ ]:
# Cell 3: 3-Mode Collapse & Ablation Audit (72L Full vs 72L No-Engram vs 36L Base Backbone)
from sparkx_loop_v03_train_env import SparkXAgenticWorkspace

workspace = SparkXAgenticWorkspace('/content/sparkx_v03_eval_workspace', model=model, tokenizer=tokenizer)
ablation_results = workspace.run_collapse_and_ablation_audit(max_new_tokens=4096)

In [ ]:
# Cell 4: Out-of-Distribution Coding & STEM Benchmark Suite (Automated Python Execution)
import re, time, torch
from sparkx_loop_v03_train_env import extract_code_from_response

model.set_loop(start=4, end=28, passes=2, use_mhc=True, use_engram=True, enable_p3=True, p3_start=12, p3_end=24)
device = next(model.base.parameters()).device

ood_benchmarks = [
    {
        'name': 'OOD_SWE_TTL_LRU_Cache',
        'prompt': (
            'Write a self-contained Python 3 class `TTLLRUCache(capacity: int)` that implements an O(1) Least-Recently-Used '
            'cache with per-key expiration timestamps. Provide:\n'
            '- `get(key: int, now: float) -> int`: returns the value if `key` exists and `now < expiry`, updating recency; otherwise removes expired key and returns `-1`.\n'
            '- `put(key: int, value: int, ttl: float, now: float) -> None`: inserts/updates `key` with expiration `now + ttl` and evicts the least-recently-used item if capacity is exceeded (after purging any expired representation of `key`).\n'
            'Think briefly in `<think>...</think>`, then output ONLY one ```python ... ``` block.'
        ),
        'test_code': (
            'c = TTLLRUCache(2)\n'
            'c.put(1, 10, ttl=5.0, now=100.0)\n'
            'c.put(2, 20, ttl=10.0, now=101.0)\n'
            'assert c.get(1, now=102.0) == 10\n'
            'c.put(3, 30, ttl=5.0, now=103.0)\n'
            'assert c.get(2, now=103.5) == -1\n'
            'assert c.get(1, now=104.9) == 10\n'
            'assert c.get(1, now=105.1) == -1\n'
            'print("TTL_LRU_OK")\n'
        ),
        'expected_stdout': 'TTL_LRU_OK',
    },
    {
        'name': 'OOD_Algorithm_MinWindowSubstring',
        'prompt': (
            'Write a self-contained Python 3 function `min_window_substring(s: str, t: str) -> str` that finds the minimum '
            'contiguous substring of `s` containing all characters of `t` (including multiplicity) in O(|s| + |t|) time using '
            'two pointers. If no such window exists, return `""`.\n'
            'Think briefly in `<think>...</think>`, then output ONLY one ```python ... ``` block.'
        ),
        'test_code': (
            'assert min_window_substring("ADOBECODEBANC", "ABC") == "BANC"\n'
            'assert min_window_substring("a", "aa") == ""\n'
            'assert min_window_substring("aaflslflsldkalskaaa", "aaa") == "aaa"\n'
            'print("MIN_WINDOW_OK")\n'
        ),
        'expected_stdout': 'MIN_WINDOW_OK',
    },
]

for bench in ood_benchmarks:
    print('\n' + '-' * 96)
    print(f"Running {bench['name']} (72L Full Loop + 32-Expert Top-7 Engram)...")
    messages = [
        {'role': 'system', 'content': 'You are Spark-X2.6-4B-En4B-A1B, an expert 72-layer reasoning and coding assistant.'},
        {'role': 'user', 'content': bench['prompt']},
    ]
    raw_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    p_ids = tokenizer(raw_prompt, return_tensors='pt').input_ids.to(device)
    t0 = time.time()
    out_ids = model.generate_cached(
        p_ids,
        max_new_tokens=4096,
        temperature=0.25,
        top_p=0.90,
        repetition_penalty=1.03,
        tokenizer=tokenizer,
        stream_print=True,
    )
    dt = max(1e-3, time.time() - t0)
    gen_ids = out_ids[0, p_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    code = extract_code_from_response(text, target_ext='.py')
    res = workspace.execute_python(f'{bench["name"]}.py', code + '\n\n' + bench['test_code'])
    passed = res['ok'] and (bench['expected_stdout'] in res['stdout'])
    print(
        f"[{bench['name']} Result] Passed={passed} | Tokens={gen_ids.numel()} ({gen_ids.numel()/dt:.1f} tok/s) | "
        f"Stdout={res['stdout'].strip()!r}"
    )
    if not passed:
        print('Stderr:', res['stderr'][:500])

In [ ]:
# Cell 5: Unclamped 32K Agentic 3D WebGL Synthesis & Live Preview
# Set MISSION_CHOICE = 'neon_horizon_ood' (out-of-distribution Cyberpunk Hovercar Sim) or 'sakura_garden'
import html
from IPython.display import HTML, display
from sparkx_loop_v03_train_env import DEFAULT_MAX_NEW_TOKENS, OOD_NEON_HORIZON_MISSION

MISSION_CHOICE = 'neon_horizon_ood'  # Options: 'neon_horizon_ood' or 'sakura_garden'

sakura_mission = (
    'Build a complete, self-contained, single-file HTML5 + Three.js 3D Interactive Voxel Cherry Blossom '
    '(Sakura) Garden webpage (`<!DOCTYPE html>` ... `</html>`) and save it as `index.html`.\n'
    'Requirements:\n'
    '1. Load Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n'
    '2. Use `THREE.InstancedMesh` to render a stepped voxel island terrain with a reflective pond, stepping stones, and red Torii gate / wooden arched bridge.\n'
    '3. Build multiple gnarled dark-wood cherry blossom trees with layered pink/blush voxel canopies and Japanese stone lanterns (`Toro`) with warm point lights.\n'
    '4. Animate at least 400 falling pink sakura petals drifting in the wind and floating on the pond.\n'
    '5. Include interactive mouse-drag orbit & scroll-wheel zoom camera controls, auto-rotate toggle, and HUD buttons for Spring Daylight / Golden Sunset / Lantern Night.'
)

chosen_prompt = OOD_NEON_HORIZON_MISSION if MISSION_CHOICE == 'neon_horizon_ood' else sakura_mission
mission_result = workspace.run_agentic_mission(
    chosen_prompt,
    target_filename='index.html',
    max_turns=2,
    max_new_tokens_per_turn=DEFAULT_MAX_NEW_TOKENS,
    run_ablation_first=False,
    use_ood_mission=(MISSION_CHOICE == 'neon_horizon_ood'),
)

if mission_result.get('html'):
    escaped = html.escape(mission_result['html'], quote=True)
    display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="600" style="border:1px solid #00e5ff; border-radius:10px;"></iframe>'))

In [ ]:
# Cell 6: Interactive 72-Layer Prompt Playground
def ask_sparkx_72l(
    prompt: str,
    system_prompt: str = 'You are Spark-X2.6-4B-En4B-A1B, a 72-layer Looped Transformer with 8-anchor 32-expert MoE-Engram.',
    max_new_tokens: int = 8192,
    temperature: float = 0.35,
    top_p: float = 0.90,
) -> str:
    model.set_loop(start=4, end=28, passes=2, use_mhc=True, use_engram=True, enable_p3=True, p3_start=12, p3_end=24)
    device = next(model.base.parameters()).device
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]
    raw_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(raw_prompt, return_tensors='pt').input_ids.to(device)
    out_ids = model.generate_cached(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.03,
        tokenizer=tokenizer,
        stream_print=True,
    )
    reply = tokenizer.decode(out_ids[0, input_ids.shape[1]:], skip_special_tokens=True)
    print('\n' + '=' * 96 + '\nMODEL RESPONSE:\n' + '=' * 96)
    print(reply)
    return reply

# Example interactive test:
_ = ask_sparkx_72l(
    'Explain why two-stream doubly-stochastic Sinkhorn Hyper-Connections (mHC) prevent signal explosion '
    'across 72 recurrent transformer visits, and provide a concise PyTorch implementation of a 2x2 closed-form Sinkhorn mixer.'
)